# Deploying NVIDIA Nemotron-3-Nano-Omni with SGLang

This notebook will walk you through how to run the `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning` model with SGLang.

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models and vision language models.

For more details on the model [click here](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16)

Prerequisites:
- NVIDIA GPU with recent drivers (≥ 64 GB VRAM for BF16) and CUDA 12.9+
- Python 3.10+

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this project on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide. 

**For H100 (BF16 model):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Cm3K626RMrQZWhaumlWMdgvXoQ)

> **Note:** SGLang currently supports the BF16 variant only. FP8 and NVFP4 support is coming soon.


## Environment setup

### Verify GPU driver

Docker images are provided for both **CUDA 12.9** and **CUDA 13.0**. Check which version your driver supports:

```shell
nvidia-smi | head -5
```

Use the CUDA version shown to pick the matching Docker image later in this notebook.

> **Note:** If your driver only supports CUDA 12.8 or below, you will need to upgrade it before proceeding:
> 
> ```shell
> # Remove the old driver (adjust 550 to match your installed version)
> sudo apt remove -y --purge 'nvidia-*-550'
> sudo apt autoremove -y
> 
> # Install a newer driver
> sudo apt update
> sudo apt install -y nvidia-driver-580
> sudo reboot
> ```
> 
> **Brev users:** Instances launched from the buttons above already include a compatible driver — skip this step.

### Install dependencies

In [ ]:
%pip install openai huggingface_hub transformers

### Launch the SGLang Docker container

The BF16 model is supported via a dedicated SGLang Docker image. Start an interactive container in a terminal:

**CUDA 13.0:**
```shell
docker run --rm -it --gpus all \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    -p 5000:5000 \
    lmsysorg/sglang:dev-cu13-nemotronh-nano-omni-reasoning-v3 \
    bash
```

**CUDA 12.9:**
```shell
docker run --rm -it --gpus all \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    -p 5000:5000 \
    lmsysorg/sglang:dev-nemotronh-nano-omni-reasoning-v3 \
    bash
```

Once inside the container, install `librosa` (required for audio processing):
```shell
pip install librosa --break-system-packages
```

## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.


In [1]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 1
GPU[0]: NVIDIA H100 PCIe


## Start SGLang Server

SGLang runs as a separate server process inside the Docker container.

### Load the BF16 version

From inside the Docker container, run:

```shell
sglang serve \
    --model-path nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
    --host 0.0.0.0 --port 5000 \
    --trust-remote-code \
    --tool-call-parser qwen3_coder \
    --reasoning-parser deepseek-r1
```

### FP8 and NVFP4 variants

SGLang support for the [FP8](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8) and [NVFP4](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4) quantized variants is coming soon. Check the model cards for updates.

## Generate responses


In [2]:
## Client Setup
from openai import OpenAI

# The model name we used when launching the server.
SERVED_MODEL_NAME = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"

BASE_URL = f"http://localhost:5000/v1"
API_KEY = "EMPTY"  # SGLang server doesn't require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"OpenAI client configured to use server at: {BASE_URL}")
print(f"Using model: {SERVED_MODEL_NAME}")

OpenAI client configured to use server at: http://localhost:5000/v1
Using model: nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16


### Simple vs streamed generation


In [3]:
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Give me 3 bullet points about SGLang."}
    ],
    temperature=0.6,
    max_tokens=512,
)
print(resp.choices[0].message.reasoning_content, resp.choices[0].message.content)
print("\n")

We need to respond with 3 bullet points about SGLang. No policy issues. Provide concise bullet points.
 - SGLang is an open‑source inference serving library designed for high‑throughput, low‑latency serving of large language models (LLMs) on GPUs and other accelerators.  
- It features a flexible, plug‑in architecture that supports models from Hugging Face, TensorRT‑LLM, and custom back‑ends, enabling easy integration with existing model pipelines.  
- SGLang provides automatic batching, dynamic scheduling, and advanced load‑balancing, allowing users to achieve significant speed‑ups and cost efficiencies compared to traditional serving frameworks.




In [5]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)


The first 5 prime numbers are:

1. 2  
2. 3  
3. 5  
4. 7  
5. 11

### Reasoning

Note: The model supports two modes - Reasoning ON (default) vs OFF. This can be toggled by setting enable_thinking to False, as shown below. 

In [6]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=256,
)
print(resp.choices[0].message.reasoning_content, resp.choices[0].message.content)
print("\n")
# Reasoning off
print("Reasoning off")
resp = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about SGLang."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(resp.choices[0].message.reasoning_content, resp.choices[0].message.content)

Reasoning on
We need to write a haiku about GPUs. 5-7-5 syllable structure. Ensure it's about GPUs. Let's craft:

Parallel cores ignite (5)
Silicon rivers flow fast (7)
Dreams rendered in light (5)

Check syllable count:

"Parallel cores ignite" => Paral-lel (3) cores (1) i-gnite (2) total 6? Wait count: "Par" is one? Let's count: Par (1) al (2) lel (3) actually "parallel" is 3 syllables: Par (1) al (2) lel (3). "cores" 1. "ignite" 2 (i-gnite). So total 3+1+2=6. Too many. Need 5.

Try "Cores ignite" (Cores=1, ig-nite=2) total 3, need 5. Add "fast": "Cores ignite fast" -> Cores (1) ig-nite (2) fast (1) =4. Need one more maybe "bright": "Cores ignite fast bright" -> bright 1. total 5.

Second line: " None


Reasoning off
Sure, I'd be happy to help! Here are three interesting facts about SGLang:

1. **SGLang is a high-performance compiler for deep learning frameworks**: SGLang is a compiler that optimizes the performance of deep learning frameworks such as TensorFlow and PyTorch. It uses 

### Tool calling

Call functions using the OpenAI Tools schema and inspect returned tool_calls.

In [4]:
# Tool calling via OpenAI tools schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print(completion.choices[0].message.reasoning_content)
print(completion.choices[0].message.tool_calls)

Okay, let's see. The user has a bill of $50 and wants to know the amount for a 15% tip. Hmm, I need to calculate 15% of 50. The tool provided is calculate_tip, which takes bill_total and tip_percentage. So, bill_total is 50 and tip_percentage is 15. I should call the function with these values. Let me make sure the parameters are integers. Yes, 50 and 15 are both integers. The function should return the tip amount. I don't need to do anything else except call the tool here.

[ChatCompletionMessageFunctionToolCall(id='call_0bf3099f8967459484b9e97d', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function', index=0)]


### Video understanding

Nemotron-3-Nano-Omni is a multimodal model that can reason over text, images, audio, and video.

The cell below demonstrates video understanding: it sends a short NVIDIA Omniverse clip to the model via the OpenAI-compatible API and asks it to describe the scene.

In [3]:
video_url = "https://blogs.nvidia.com/wp-content/uploads/2023/04/nvidia-studio-itns-wk53-scene-in-omniverse-1280w.mp4"

resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what happens in this video."},
                {"type": "video_url", "video_url": {"url": video_url}},
            ],
        }
    ],
    temperature=0.6,
    max_tokens=2048,
)

print(resp.choices[0].message.content)

The video displays a screen recording of a 3D game development environment, likely Unreal Engine, showcasing a winter scene.

*   **Scene Setup:** In the center of the viewport is a small, circular island or platform covered in snow. A rustic wooden cabin sits near the center, surrounded by tall, dark pine trees. The ground is textured with snow, rocks, logs, and a low stone wall. To the right, the software's interface is visible, showing an "Outliner" with scene hierarchy and a "Details" panel with material settings.
*   **Camera Movement:** The camera begins with a wider shot, slowly zooming in and rotating around the cabin. As it gets closer, the details of the snowy roof on the cabin and the surrounding vegetation become clearer.
*   **Character:** A humanoid character model, dressed in what appears to be armor or medieval clothing, is standing on the snowy ground near the cabin, next to a table and some barrels.
*   **Progression:** The camera continues to move, panning left and t

### Controlling Reasoning Budget

The `reasoning_budget` parameter allows you to limit the length of the model's reasoning trace. When the reasoning output reaches the specified token budget, the model will attempt to gracefully end the reasoning at the next newline character. 

If no newline is encountered within 500 tokens after reaching the budget threshold, the reasoning trace will be forcibly terminated at `reasoning_budget + 500` tokens to prevent excessive generation.


In [8]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. first call chat completion to get reasoning content
        response = self.client.chat.completions.create(
            model=model, 
            messages=messages, 
            max_tokens=reasoning_budget, 
            **kwargs
        )
        
        reasoning_content = response.choices[0].message.reasoning_content or ""
        
        if "</think>" not in reasoning_content:
            # reasoning content is too long, closed with a period (.)
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"
        
        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        
        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # 2. append reasoning content to messages and call completion
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )
        
        response = self.client.completions.create(
            model=model, 
            prompt=prompt, 
            max_tokens=remaining_tokens, 
            **kwargs
        )

        response_data = {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }
        return response_data

/home/shadeform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Client
SERVED_MODEL_NAME = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
client = ThinkingBudgetClient(
    base_url="http://127.0.0.1:5000/v1",
    api_key="null",
    tokenizer_name_or_path=SERVED_MODEL_NAME
)

The repository nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 .
 You can inspect the repository content at https://hf.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


In [10]:
resp = client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"], "\nContent:", resp["content"])

Reasoning: We need to respond with a haiku about GPUs. A haiku is 5-7-5 syllable structure. Provide a haiku. Ensure it's about GPUs. Probably 5 syllables first line, 7 second, 5 third. Let's craft:

"Silicon hearts blaze" – that's 5? "Sil-i-con hearts blaze": Sil(1) i(2) con(3) hearts(4) blaze(5). Good.

"Parallel rivers of light" - count: Par(1) allel? Actually "Par" "a" "lel" =. 
Content: 
Silicon hearts blaze  
Parallel rivers of light  
Dreams rendered swift


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the SGLang server process in the terminal where it was started (`Ctrl+C`).
2. Optionally run the next cell to clear notebook-side CUDA cache.
3. If needed, restart the kernel to ensure a clean state.

In [8]:
import gc
import torch

# Optional notebook-side cleanup after server shutdown
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")
print("Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.")

Notebook-side CUDA cache cleanup complete.
Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.
